In [1]:
# colab_02b_kalibrasyon.py  (v2 - leave-one-origin-out secim)
# Val'de katsayi ogrenilir, val ICINDE LOO ile yapi secilir, TEST'e TEK SEFER uygulanir.
# Cikti: model/kalibrasyon.json | model/kalibrasyon.png

import os, json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
# ------------------------------------------------------------------
# 1. AYARLAR (colab_02 ile BIREBIR AYNI)
# ------------------------------------------------------------------
DIZIN = "/content/drive/MyDrive/Colab Notebooks/datasets/rossman"
HAZIRLIK = f"{DIZIN}/hazirlik"
CIKTI = f"{DIZIN}/model"

EMB_BOYUT, GRU_BIRIM, COZUCU_BIRIM = 24, 128, 128
DROPOUT, L2 = 0.3, 1e-5
QUANTILES = [0.10, 0.50, 0.90]
SEEDLER = [42, 1337, 2024]

HEDEF_KAPSAMA = 80.0
KUYRUK = 0.10
BASITLIK_TOLERANSI = 0.3   # LOO farki bu kadarin altindaysa DAHA BASIT yapi secilir

print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

TF: 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ------------------------------------------------------------------
# 2. VERI
# ------------------------------------------------------------------
meta = json.load(open(f"{HAZIRLIK}/meta.json"))
GECMIS, UFUK = meta["gecmis"], meta["ufuk"]
N_MAGAZA = meta["n_magaza"]
magaza_ort = np.array(meta["magaza_ort"], dtype=np.float32)
magaza_std = np.array(meta["magaza_std"], dtype=np.float32)

def yukle(ad):
    d = np.load(f"{HAZIRLIK}/nn_{ad}.npz")
    return {k: d[k] for k in d.files}

val, test = yukle("val"), yukle("test")
N_GECMIS_KANAL = val["X_gecmis"].shape[-1]
N_GELECEK_KANAL = val["X_gelecek"].shape[-1]
assert N_GECMIS_KANAL == len(meta["gecmis_kanal"]), "Gecmis kanal uyusmazligi!"
assert N_GELECEK_KANAL == len(meta["gelecek_kanal"]), "Gelecek kanal uyusmazligi!"
print(f"val {val['X_gecmis'].shape} | test {test['X_gecmis'].shape}")

val (5575, 56, 5) | test (1115, 56, 5)


In [4]:

# ------------------------------------------------------------------
# 3. MODEL (colab_02'den birebir)
# ------------------------------------------------------------------
def model_kur():
    g_in = keras.Input(shape=(GECMIS, N_GECMIS_KANAL), name="gecmis")
    f_in = keras.Input(shape=(UFUK, N_GELECEK_KANAL), name="gelecek")
    s_in = keras.Input(shape=(), dtype="int32", name="magaza")

    emb = layers.Embedding(N_MAGAZA, EMB_BOYUT,
                           embeddings_regularizer=keras.regularizers.l2(L2),
                           name="magaza_embedding")(s_in)
    emb = layers.Flatten(name="magaza_vektor")(emb)

    h = layers.GRU(GRU_BIRIM, return_sequences=True,
                   kernel_regularizer=keras.regularizers.l2(L2))(g_in)
    h = layers.Dropout(DROPOUT)(h)
    h = layers.GRU(GRU_BIRIM, kernel_regularizer=keras.regularizers.l2(L2))(h)
    h = layers.Dropout(DROPOUT)(h)

    baglam = layers.Concatenate()([h, emb])
    baglam = layers.Dense(COZUCU_BIRIM, activation="relu")(baglam)
    baglam_seq = layers.RepeatVector(UFUK)(baglam)

    d = layers.Concatenate()([f_in, baglam_seq])
    d = layers.GRU(COZUCU_BIRIM, return_sequences=True,
                   kernel_regularizer=keras.regularizers.l2(L2))(d)
    d = layers.Dropout(DROPOUT)(d)
    d = layers.TimeDistributed(layers.Dense(64, activation="relu"))(d)
    cikis = layers.TimeDistributed(layers.Dense(len(QUANTILES)), name="quantiles")(d)
    return keras.Model([g_in, f_in, s_in], cikis, name="rossmann_global")

def girdi_paketle(d):
    return {"gecmis": d["X_gecmis"], "gelecek": d["X_gelecek"], "magaza": d["X_magaza"]}

modeller = []
for sd in SEEDLER:
    m = model_kur()
    m.load_weights(f"{CIKTI}/global_seed{sd}.weights.h5")
    modeller.append(m)
print(f"{len(modeller)} model yuklendi | parametre: {modeller[0].count_params():,}")

def tahmin_al(d):
    return np.mean([m.predict(girdi_paketle(d), batch_size=512, verbose=0)
                    for m in modeller], axis=0)

p_val, p_test = tahmin_al(val), tahmin_al(test)
print("tahminler hazir:", p_val.shape, p_test.shape)

3 model yuklendi | parametre: 310,155
tahminler hazir: (5575, 28, 3) (1115, 28, 3)


In [5]:
# ------------------------------------------------------------------
# 4. YARDIMCILAR
# ------------------------------------------------------------------
def ters_olcek(y_olcek, magaza_idx):
    ort = magaza_ort[magaza_idx][:, None]
    std = magaza_std[magaza_idx][:, None]
    return np.expm1(y_olcek * std + ort)

def artiklar(p, d):
    p10, p50, p90 = p[..., 0], p[..., 1], p[..., 2]
    y = d["y"]
    r = (p50 - y) / np.maximum(p50 - p10, 1e-6)
    s = (y - p50) / np.maximum(p90 - p50, 1e-6)
    return r, s

def kfy(k):
    """k'yi (1,28) yayinlanabilir hale getir"""
    a = np.asarray(k, dtype=np.float32)
    return a[None, :] if a.ndim == 1 else a

def genislet(p, k_alt, k_ust):
    p10, p50, p90 = p[..., 0], p[..., 1], p[..., 2]
    return p50 - kfy(k_alt) * (p50 - p10), p50, p50 + kfy(k_ust) * (p90 - p50)

def kapsama_maske(p, d, maske, k_alt=1.0, k_ust=1.0):
    a, _, u = genislet(p, k_alt, k_ust)
    icinde = (d["y"] >= a) & (d["y"] <= u)
    return float(100.0 * icinde[maske].mean())

def kapsama_ufuk(p, d, maske, k_alt=1.0, k_ust=1.0):
    a, _, u = genislet(p, k_alt, k_ust)
    icinde = (d["y"] >= a) & (d["y"] <= u)
    return np.array([100.0 * icinde[:, h][maske[:, h]].mean() for h in range(UFUK)])

def kapsama_magaza(p, d, maske, k_alt=1.0, k_ust=1.0):
    a, _, u = genislet(p, k_alt, k_ust)
    icinde = (d["y"] >= a) & (d["y"] <= u)
    no = d["magaza_no"]
    out = []
    for mno in np.unique(no):
        sec = (no == mno); mk = maske[sec]
        if mk.sum() == 0: continue
        out.append(100.0 * icinde[sec][mk].mean())
    return np.array(out)

def band_eur(p, d, maske, k_alt=1.0, k_ust=1.0):
    a, _, u = genislet(p, k_alt, k_ust)
    a_e = np.maximum(ters_olcek(a, d["X_magaza"]), 0.0)
    u_e = ters_olcek(u, d["X_magaza"])
    return float((u_e - a_e)[maske].mean())

def pinball_eur(p, d, maske, k_alt=1.0, k_ust=1.0):
    a, p50s, u = genislet(p, k_alt, k_ust)
    g = d["y_ham"]; toplam = 0.0
    for q, ps in zip(QUANTILES, [a, p50s, u]):
        t = np.maximum(ters_olcek(ps, d["X_magaza"]), 0.0)
        hata = g - t
        toplam += np.mean(np.maximum(q * hata, (q - 1.0) * hata)[maske])
    return float(toplam / len(QUANTILES))

In [6]:
# ------------------------------------------------------------------
# 5. KATSAYI OGRENME - ANALITIK
#    y < p50 - k*(p50-p10)  <=>  r > k   =>  k = r'nin 90. persentili
# ------------------------------------------------------------------
def k_ogren(r, s, maske, yapi):
    P = 100 * (1 - KUYRUK)
    if yapi == "A":
        return 1.0, 1.0
    if yapi == "B":
        k = float(np.percentile(np.maximum(r, s)[maske], HEDEF_KAPSAMA)); return k, k
    if yapi == "C":
        return float(np.percentile(r[maske], P)), float(np.percentile(s[maske], P))
    if yapi in ("D", "D_duz"):
        ka = np.array([np.percentile(r[:, h][maske[:, h]], P) if maske[:, h].sum() > 0 else 1.0
                       for h in range(UFUK)], dtype=np.float32)
        ku = np.array([np.percentile(s[:, h][maske[:, h]], P) if maske[:, h].sum() > 0 else 1.0
                       for h in range(UFUK)], dtype=np.float32)
        if yapi == "D_duz":   # ufuk etkisi GERCEK ise dogruya uyar, gurultu ise sabitlesir
            h = np.arange(UFUK)
            ka = np.polyval(np.polyfit(h, ka, 1), h).astype(np.float32)
            ku = np.polyval(np.polyfit(h, ku, 1), h).astype(np.float32)
        return ka, ku
    raise ValueError(yapi)

YAPILAR = ["A", "B", "C", "D_duz", "D"]      # basitten karmasiga
ETIKET = {"A": "A_kalibrasyonsuz", "B": "B_tek_k", "C": "C_iki_tarafli",
          "D_duz": "D_duz_ufuk_dogrusal", "D": "D_ufuk_serbest"}

mv = val["maske"].astype(bool)
mt = test["maske"].astype(bool)
r_val, s_val = artiklar(p_val, val)
r_test, s_test = artiklar(p_test, test)

for ad, p in [("val", p_val), ("test", p_test)]:
    capraz = int(((p[..., 0] > p[..., 1]) | (p[..., 1] > p[..., 2])).sum())
    print(f"{ad}: capraz gecen quantile {capraz} / {p[..., 0].size}")

val: capraz gecen quantile 0 / 156100
test: capraz gecen quantile 0 / 31220


In [7]:
# ------------------------------------------------------------------
# 6. TESHIS (VAL) - ufuk deseni gercek mi, takvim mi?
# ------------------------------------------------------------------
print("\n" + "="*62); print("KALIBRASYON ONCESI (VAL)"); print("="*62)
print(f"alt sinir asimi : {100.0*(r_val[mv] > 1.0).mean():.2f}%  (hedef 10.0)")
print(f"ust sinir asimi : {100.0*(s_val[mv] > 1.0).mean():.2f}%  (hedef 10.0)")
print(f"toplam kapsama  : {kapsama_maske(p_val, val, mv):.2f}%  (hedef {HEDEF_KAPSAMA})")

kap_uf_once = kapsama_ufuk(p_val, val, mv)
yayilim = float(kap_uf_once.max() - kap_uf_once.min())
egim = float(np.polyfit(np.arange(UFUK), kap_uf_once, 1)[0])
print(f"ufuk kapsamasi  : min {kap_uf_once.min():.1f}% (h={int(kap_uf_once.argmin())+1}) "
      f"| max {kap_uf_once.max():.1f}% (h={int(kap_uf_once.argmax())+1}) | yayilim {yayilim:.1f}p")
print(f"ufka gore egim  : {egim:+.3f} puan/gun  "
      f"<- gercek ufuk etkisi MONOTON olmali; sifira yakinsa desen takvimden geliyor")

origin_val = val["origin_idx"]
originler = np.unique(origin_val)
print(f"val origin sayisi: {len(originler)}  -> her h dilimi icin efektif ornek "
      f"~{len(originler)} TARIH (1115 magaza ayni gun korele)")


KALIBRASYON ONCESI (VAL)
alt sinir asimi : 10.51%  (hedef 10.0)
ust sinir asimi : 11.64%  (hedef 10.0)
toplam kapsama  : 77.85%  (hedef 80.0)
ufuk kapsamasi  : min 66.5% (h=10) | max 83.7% (h=15) | yayilim 17.2p
ufka gore egim  : +0.014 puan/gun  <- gercek ufuk etkisi MONOTON olmali; sifira yakinsa desen takvimden geliyor
val origin sayisi: 5  -> her h dilimi icin efektif ornek ~5 TARIH (1115 magaza ayni gun korele)


In [8]:
# ------------------------------------------------------------------
# 7. SECIM: LEAVE-ONE-ORIGIN-OUT (val icinde, TEST'e bakilmadan)
# ------------------------------------------------------------------
print("\n" + "="*62); print("LEAVE-ONE-ORIGIN-OUT (val ici, 5 kat)"); print("="*62)
print("Katsayi 4 origin'den ogrenilir, 5.'sinde olculur. Kalibrasyon yeni bir TARIHE genelleniyor mu?\n")

loo = {}
print(f"{'yapi':<22}" + "".join(f"{f'kat{i+1}':>9}" for i in range(len(originler))) + f"{'ort|hata|':>11}")
for yapi in YAPILAR:
    kapsamalar = []
    for o in originler:
        egit_m = mv & (origin_val != o)[:, None]
        olc_m  = mv & (origin_val == o)[:, None]
        ka, ku = k_ogren(r_val, s_val, egit_m, yapi)
        kapsamalar.append(kapsama_maske(p_val, val, olc_m, ka, ku))
    kapsamalar = np.array(kapsamalar)
    hata = float(np.mean(np.abs(kapsamalar - HEDEF_KAPSAMA)))
    loo[yapi] = {"kat_kapsama": kapsamalar.tolist(), "ort_mutlak_hata": hata}
    print(f"{ETIKET[yapi]:<22}" + "".join(f"{k:>8.1f}%" for k in kapsamalar) + f"{hata:>10.2f}p")

en_iyi_hata = min(loo[y]["ort_mutlak_hata"] for y in YAPILAR)
secilen = next(y for y in YAPILAR
               if loo[y]["ort_mutlak_hata"] <= en_iyi_hata + BASITLIK_TOLERANSI)
gerekce = (f"LOO ortalama mutlak kapsama hatasi en dusuk {en_iyi_hata:.2f}p; "
           f"{BASITLIK_TOLERANSI}p tolerans icinde kalan EN BASIT yapi secildi -> {ETIKET[secilen]}")
print(f"\nSECIM: {ETIKET[secilen]}\nGerekce: {gerekce}")

# secilen yapinin katsayisi TUM val'den ogrenilir
k_alt_sec, k_ust_sec = k_ogren(r_val, s_val, mv, secilen)
if np.asarray(k_alt_sec).ndim == 0:
    print(f"k_alt {k_alt_sec:.3f} | k_ust {k_ust_sec:.3f}")
else:
    print(f"k_alt h1 {k_alt_sec[0]:.2f} .. h28 {k_alt_sec[-1]:.2f} | "
          f"k_ust h1 {k_ust_sec[0]:.2f} .. h28 {k_ust_sec[-1]:.2f}")

print("\nTUM val'den ogrenilen katsayilarla VAL (kendi uzerinde - referans):")
print(f"{'aday':<22}{'kapsama':>9}{'band EUR':>11}{'pinball':>10}")
tum_k = {}
for yapi in YAPILAR:
    ka, ku = k_ogren(r_val, s_val, mv, yapi); tum_k[yapi] = (ka, ku)
    print(f"{ETIKET[yapi]:<22}{kapsama_maske(p_val, val, mv, ka, ku):>8.2f}%"
          f"{band_eur(p_val, val, mv, ka, ku):>11.0f}{pinball_eur(p_val, val, mv, ka, ku):>10.1f}")



LEAVE-ONE-ORIGIN-OUT (val ici, 5 kat)
Katsayi 4 origin'den ogrenilir, 5.'sinde olculur. Kalibrasyon yeni bir TARIHE genelleniyor mu?

yapi                       kat1     kat2     kat3     kat4     kat5  ort|hata|
A_kalibrasyonsuz          77.9%    77.2%    78.8%    77.8%    77.5%      2.15p
B_tek_k                   80.2%    79.2%    81.1%    79.9%    79.6%      0.52p
C_iki_tarafli             80.2%    79.0%    81.1%    79.5%    79.3%      0.71p
D_duz_ufuk_dogrusal       79.6%    78.5%    81.0%    78.3%    78.5%      1.20p
D_ufuk_serbest            76.4%    74.8%    78.8%    74.3%    76.5%      3.86p

SECIM: B_tek_k
Gerekce: LOO ortalama mutlak kapsama hatasi en dusuk 0.52p; 0.3p tolerans icinde kalan EN BASIT yapi secildi -> B_tek_k
k_alt 1.051 | k_ust 1.051

TUM val'den ogrenilen katsayilarla VAL (kendi uzerinde - referans):
aday                    kapsama   band EUR   pinball
A_kalibrasyonsuz         77.85%       1600     171.6
B_tek_k                  80.00%       1682     171.4
C

In [9]:
# ------------------------------------------------------------------
# 8. TEST - TEK SEFER
# ------------------------------------------------------------------
print("\n" + "="*62)
print(f"TEST ({meta['test_hedef'][0]}..{meta['test_hedef'][1]}) - kalibrasyon TEST'te ogrenilmedi")
print("="*62)

kap_t_once  = kapsama_maske(p_test, test, mt)
kap_t_sonra = kapsama_maske(p_test, test, mt, k_alt_sec, k_ust_sec)
b_once, b_sonra = band_eur(p_test, test, mt), band_eur(p_test, test, mt, k_alt_sec, k_ust_sec)
pin_once, pin_sonra = pinball_eur(p_test, test, mt), pinball_eur(p_test, test, mt, k_alt_sec, k_ust_sec)

print(f"kapsama      : {kap_t_once:.2f}%  ->  {kap_t_sonra:.2f}%   (hedef {HEDEF_KAPSAMA})")
print(f"band (EUR)   : {b_once:.0f}  ->  {b_sonra:.0f}   (+{100*(b_sonra/b_once-1):.1f}%)  <- BEDEL")
print(f"pinball (EUR): {pin_once:.1f}  ->  {pin_sonra:.1f}")
print(f"alt asim     : {100.0*(r_test > kfy(k_alt_sec))[mt].mean():.2f}%  (hedef 10.0)")
print(f"ust asim     : {100.0*(s_test > kfy(k_ust_sec))[mt].mean():.2f}%  (hedef 10.0)")
print(f"\nval->test kalan bosluk: {HEDEF_KAPSAMA - kap_t_sonra:.2f} puan  "
       "(val'de kapanip test'te kapanmayan kisim = DONEM KAYMASI, kalibrasyonla kapatilamaz)")

kap_uf_t_once  = kapsama_ufuk(p_test, test, mt)
kap_uf_t_sonra = kapsama_ufuk(p_test, test, mt, k_alt_sec, k_ust_sec)
kap_uf_v_sonra = kapsama_ufuk(p_val, val, mv, k_alt_sec, k_ust_sec)
mg_once  = kapsama_magaza(p_test, test, mt)
mg_sonra = kapsama_magaza(p_test, test, mt, k_alt_sec, k_ust_sec)

print(f"\nmagaza bazli kapsama (TEST):")
print(f"  once : p5 {np.percentile(mg_once,5):.1f}% | p50 {np.percentile(mg_once,50):.1f}% | p95 {np.percentile(mg_once,95):.1f}%")
print(f"  sonra: p5 {np.percentile(mg_sonra,5):.1f}% | p50 {np.percentile(mg_sonra,50):.1f}% | p95 {np.percentile(mg_sonra,95):.1f}%")
print(f"  kapsamasi %70 altinda kalan magaza: {int((mg_once<70).sum())} -> {int((mg_sonra<70).sum())}")

print("\n(bilgi amacli - secim LOO ile val'de yapildi, buraya bakilip degistirilmeyecek)")
for yapi in YAPILAR:
    ka, ku = tum_k[yapi]
    print(f"  {ETIKET[yapi]:<22}: TEST {kapsama_maske(p_test, test, mt, ka, ku):.2f}%")



TEST (2015-07-04..2015-07-31) - kalibrasyon TEST'te ogrenilmedi
kapsama      : 75.00%  ->  77.38%   (hedef 80.0)
band (EUR)   : 1613  ->  1697   (+5.2%)  <- BEDEL
pinball (EUR): 180.2  ->  179.7
alt asim     : 11.59%  (hedef 10.0)
ust asim     : 11.03%  (hedef 10.0)

val->test kalan bosluk: 2.62 puan  (val'de kapanip test'te kapanmayan kisim = DONEM KAYMASI, kalibrasyonla kapatilamaz)

magaza bazli kapsama (TEST):
  once : p5 50.0% | p50 75.0% | p95 91.7%
  sonra: p5 54.2% | p50 79.2% | p95 95.8%
  kapsamasi %70 altinda kalan magaza: 278 -> 217

(bilgi amacli - secim LOO ile val'de yapildi, buraya bakilip degistirilmeyecek)
  A_kalibrasyonsuz      : TEST 75.00%
  B_tek_k               : TEST 77.38%
  C_iki_tarafli         : TEST 77.33%
  D_duz_ufuk_dogrusal   : TEST 76.15%
  D_ufuk_serbest        : TEST 76.59%


In [10]:
# ------------------------------------------------------------------
# 9. GRAFIK
# ------------------------------------------------------------------
fig, ax = plt.subplots(2, 2, figsize=(13, 9))
h = np.arange(1, UFUK + 1)

ax[0,0].plot(h, kap_uf_once, "o--", ms=3, label="VAL kalibrasyonsuz")
ax[0,0].plot(h, kap_uf_v_sonra, "o-", ms=3, label="VAL kalibre")
ax[0,0].plot(h, kap_uf_t_once, "s--", ms=3, label="TEST kalibrasyonsuz")
ax[0,0].plot(h, kap_uf_t_sonra, "s-", ms=3, label="TEST kalibre")
ax[0,0].axhline(HEDEF_KAPSAMA, color="k", lw=1, ls=":")
ax[0,0].set_xlabel("ufuk (gun)"); ax[0,0].set_ylabel("gerceklesen kapsama %")
ax[0,0].set_title("Nominal %80 vs gerceklesen"); ax[0,0].legend(fontsize=7); ax[0,0].grid(alpha=.3)

ax[0,1].hist(mg_once, bins=40, alpha=.6, label="kalibrasyonsuz")
ax[0,1].hist(mg_sonra, bins=40, alpha=.6, label="kalibre")
ax[0,1].axvline(HEDEF_KAPSAMA, color="k", lw=1, ls=":")
ax[0,1].set_xlabel("magaza kapsamasi %"); ax[0,1].set_ylabel("magaza sayisi")
ax[0,1].set_title("TEST - magaza bazli dagilim"); ax[0,1].legend(fontsize=8); ax[0,1].grid(alpha=.3)

for yapi in ["B", "C", "D_duz", "D"]:
    ax[1,0].plot(range(1, len(originler)+1), loo[yapi]["kat_kapsama"], "o-", ms=4,
                 label=f"{ETIKET[yapi]} ({loo[yapi]['ort_mutlak_hata']:.2f}p)")
ax[1,0].plot(range(1, len(originler)+1), loo["A"]["kat_kapsama"], "k--", ms=4, label="A (ham)")
ax[1,0].axhline(HEDEF_KAPSAMA, color="k", lw=1, ls=":")
ax[1,0].set_xlabel("disarida birakilan origin"); ax[1,0].set_ylabel("kapsama %")
ax[1,0].set_title("Leave-one-origin-out (secim burada yapildi)"); ax[1,0].legend(fontsize=7); ax[1,0].grid(alpha=.3)

for yapi in ["C", "D"]:
    ka, ku = tum_k[yapi]
    ax[1,1].plot(h, np.broadcast_to(np.asarray(ka, dtype=float).reshape(-1), (UFUK,)),
                 "o-", ms=3, label=f"{ETIKET[yapi]} k_alt")
    ax[1,1].plot(h, np.broadcast_to(np.asarray(ku, dtype=float).reshape(-1), (UFUK,)),
                 "s--", ms=3, label=f"{ETIKET[yapi]} k_ust")
ax[1,1].axhline(1.0, color="k", lw=1, ls=":")
ax[1,1].set_xlabel("ufuk (gun)"); ax[1,1].set_ylabel("genisletme katsayisi")
ax[1,1].set_title("Katsayilar: serbest ufuk deseni monoton DEGIL"); ax[1,1].legend(fontsize=7); ax[1,1].grid(alpha=.3)

plt.tight_layout(); plt.savefig(f"{CIKTI}/kalibrasyon.png", dpi=130)
print(f"\ngrafik -> {CIKTI}/kalibrasyon.png")


grafik -> /content/drive/MyDrive/Colab Notebooks/datasets/rossman/model/kalibrasyon.png


In [11]:
# ------------------------------------------------------------------
# 10. KAYIT
# ------------------------------------------------------------------
def jsonla(k):
    return k.tolist() if isinstance(k, np.ndarray) else float(k)

kal = {
    "yontem": "olcekli uzayda normalize artik persentili; yapi secimi val ici leave-one-origin-out",
    "hedef_kapsama": HEDEF_KAPSAMA,
    "secilen": ETIKET[secilen], "secim_kodu": secilen, "secim_gerekcesi": gerekce,
    "k_alt": jsonla(k_alt_sec), "k_ust": jsonla(k_ust_sec),
    "val_teshis": {"ufuk_yayilim": yayilim, "ufuk_egim_puan_gun": egim,
                   "origin_sayisi": int(len(originler)),
                   "ufuk_kapsama_once": kap_uf_once.tolist(),
                   "ufuk_kapsama_sonra": kap_uf_v_sonra.tolist()},
    "loo": {ETIKET[y]: loo[y] for y in YAPILAR},
    "adaylar_tum_val": {ETIKET[y]: {"k_alt": jsonla(tum_k[y][0]), "k_ust": jsonla(tum_k[y][1]),
                                    "val_kapsama": kapsama_maske(p_val, val, mv, *tum_k[y]),
                                    "test_kapsama": kapsama_maske(p_test, test, mt, *tum_k[y])}
                        for y in YAPILAR},
    "test": {"kapsama_once": kap_t_once, "kapsama_sonra": kap_t_sonra,
             "kalan_bosluk_puan": HEDEF_KAPSAMA - kap_t_sonra,
             "band_eur_once": b_once, "band_eur_sonra": b_sonra,
             "band_artis_yuzde": 100*(b_sonra/b_once-1),
             "pinball_once": pin_once, "pinball_sonra": pin_sonra,
             "ufuk_kapsama_once": kap_uf_t_once.tolist(),
             "ufuk_kapsama_sonra": kap_uf_t_sonra.tolist(),
             "magaza_kapsama_once_p50": float(np.percentile(mg_once, 50)),
             "magaza_kapsama_sonra_p50": float(np.percentile(mg_sonra, 50)),
             "magaza_70_alti_once": int((mg_once < 70).sum()),
             "magaza_70_alti_sonra": int((mg_sonra < 70).sum())},
}
with open(f"{CIKTI}/kalibrasyon.json", "w") as f:
    json.dump(kal, f, indent=2)
print(f"kaydedildi -> {CIKTI}/kalibrasyon.json")
print("colab_03 bu dosyadan k_alt/k_ust okuyup tahmin tablosuna uygulayacak.")

kaydedildi -> /content/drive/MyDrive/Colab Notebooks/datasets/rossman/model/kalibrasyon.json
colab_03 bu dosyadan k_alt/k_ust okuyup tahmin tablosuna uygulayacak.
